# Highlights de goles

Este cuaderno pasa un partido grabado por **T-DEED**, el modelo de acciones ya entrenado, y
monta **un clip por gol**: la jugada que lo trae (20 s antes) y la celebración (10 s
después). Al final pega los clips en un vídeo de highlights.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*. En CPU
también funciona, pero cada ventana de 8 s tarda unos 25 s: un partido entero se va a horas.

**Qué necesitas**
- El modelo `tdeed-snb.onnx` (50 MB) en tu Drive. Antes de usarlo se comprueba que su
  sha256 sea el de su ficha.
- El vídeo del partido, apaisado, en tu Drive o subido aquí.

Ejecuta las celdas de arriba abajo. Cada una comprueba lo suyo y se para con un mensaje
claro si algo no sirve, así que no se puede llegar al final con algo a medias.

> ⚠️ **Licencia.** T-DEED es GPL-3.0 y sus pesos se afinaron sobre SoccerNet, que es solo
> para investigación. Lo que salga de aquí sirve para analizar y para enseñarlo dentro
> del equipo: **no se publica ni se vende** mientras el modelo sea T-DEED.

## 1 · Entorno

Clona el repo `football-ai-analysis` e instala su paquete y el onnxruntime que casa con la
GPU:

- **El driver manda.** Desde la 1.27, `onnxruntime-gpu` va con CUDA 13, y hasta la 1.26
  con CUDA 12. Si no casa con el driver, la sesión no falla: cae a la CPU sin avisar. Por
  eso se lee la CUDA del driver en `nvidia-smi` y se instala la versión que le toca.
- **Las librerías de NVIDIA van con el paquete**, como ruedas de pip (extras
  `[cuda,cudnn]`). Así no depende de la CUDA que traiga Colab.

Si el repo es privado, guarda un token de GitHub de solo lectura en los **Secretos** de
Colab (el icono de la llave) con el nombre `GITHUB_TOKEN` y dale acceso al cuaderno. El
token no se escribe en ninguna celda ni en ningún log.

In [ ]:
import base64
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jhquihuiri7/football-ai-analysis"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}

EN_COLAB = "google.colab" in sys.modules


def corre(orden, entorno=None, que=None):
    # Lanza una orden y para aquí mismo si falla, enseñando el final de su salida.
    salida = subprocess.run(orden, capture_output=True, text=True, check=False, env=entorno)
    if salida.returncode != 0:
        print(salida.stdout[-2000:], salida.stderr[-2000:])
        raise SystemExit(f"falló: {que or ' '.join(map(str, orden))}")
    return salida.stdout


try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
    TIENE_GPU = gpu.returncode == 0
except FileNotFoundError:  # sin driver de NVIDIA no hay ni nvidia-smi
    gpu, TIENE_GPU = None, False
print("GPU        :", gpu.stdout.strip() if TIENE_GPU else "ninguna (irá por CPU: unos 25 s por ventana)")

if EN_COLAB:
    REPO = Path("/content/football-ai-analysis")
    entorno = dict(os.environ)
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:  # sin secreto o sin acceso: se intenta como repo público
        token = None
    if token:
        # El token viaja en la configuración del entorno de git: ni en la orden, ni en el
        # .git/config del clon, ni en lo que se imprime si falla.
        basico = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        entorno.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0=f"Authorization: Basic {basico}",
        )
    corre(["rm", "-rf", str(REPO)])
    corre(["git", "clone", "-q", "--depth", "1", "--branch", RAMA, REPO_URL, str(REPO)],
          entorno, que=f"clonar {REPO_URL} ({RAMA}). ¿Es privado y falta el secreto GITHUB_TOKEN?")

    # Qué onnxruntime toca, según la CUDA que admite el driver. Se lee del clon: el paquete
    # todavía no está instalado.
    sys.path.insert(0, str(REPO))
    from fanalysis.entorno import driver_cuda_version, onnxruntime_requirement

    cabecera = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout
    CUDA_DRIVER = driver_cuda_version(cabecera) if TIENE_GPU else None
    REQUISITO_ORT = onnxruntime_requirement(CUDA_DRIVER)
    # Las dos ruedas de onnxruntime instalan el mismo módulo y se pisan: fuera las dos antes.
    corre([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime", "onnxruntime-gpu"])
    corre([sys.executable, "-m", "pip", "install", "-q", f"{REPO}[plots]", REQUISITO_ORT])
    print("CUDA driver:", ".".join(map(str, CUDA_DRIVER)) if CUDA_DRIVER else "sin GPU",
          "→", REQUISITO_ORT)
else:
    # En local, el cuaderno vive en notebooks/ y el paquete ya está instalado con uv.
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

import fanalysis
import onnxruntime as ort

print("repo       :", REPO, "· commit", corre(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"]).strip())
print("fanalysis  :", fanalysis.__version__, "· onnxruntime", ort.__version__)
print("proveedores:", ", ".join(ort.get_available_providers()))

## 2 · Qué analizar

- **El tramo** sirve para probar rápido con un trozo del partido, o para partir uno largo
  en dos sesiones. `0` y `0` es el vídeo entero.
- **Los umbrales** son los medidos en la línea base de T-DEED: a 0,2 acertó 6 goles de 6,
  y el más flojo salió a 0,255.
- **Exigir remate:** un gol entra en los highlights solo si lleva un tiro o un cabezazo
  en los 2,5 s anteriores. Los 10 goles de la línea base lo llevaban. Los goles sin remate
  se enseñan igual, y en la revisión se pueden meter a mano.
- **El salto** son los frames entre el inicio de dos ventanas. Con 25 cada frame se
  promedia en cuatro ventanas, como en T-DEED. Con 50 va el doble de rápido y promedia en
  dos.

In [ ]:
#@title Parámetros
VIDEO = "/content/drive/MyDrive/football-ai/partidos/partido.mp4"  #@param {type:"string"}
MODELO_ONNX = "/content/drive/MyDrive/football-ai/models/tdeed-snb.onnx"  #@param {type:"string"}
SALIDA = "/content/drive/MyDrive/football-ai/highlights"  #@param {type:"string"}
#@markdown **Tramo**, en segundos (0 y 0 = el vídeo entero)
INICIO_S = 0  #@param {type:"number"}
FIN_S = 0  #@param {type:"number"}
#@markdown **Goles**
UMBRAL_GOL = 0.2  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
UMBRAL_ACCIONES = 0.2  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
EXIGIR_REMATE = True  #@param {type:"boolean"}
#@markdown **Antes y después de cada gol**, en segundos
SEGUNDOS_ANTES = 20  #@param {type:"number"}
SEGUNDOS_DESPUES = 10  #@param {type:"number"}
#@markdown **Rendimiento**
SALTO_FRAMES = 25  #@param {type:"integer"}
RECALCULAR = False  #@param {type:"boolean"}

In [ ]:
from fanalysis.goals import GoalError
from fanalysis.spotting import check_threshold

# Lo que se puede comprobar sin abrir nada, se comprueba ya.
for nombre, valor in (("umbral de gol", UMBRAL_GOL), ("umbral de acciones", UMBRAL_ACCIONES)):
    check_threshold(valor, nombre)
if SEGUNDOS_ANTES < 0 or SEGUNDOS_DESPUES < 0 or SEGUNDOS_ANTES + SEGUNDOS_DESPUES <= 0:
    raise GoalError("los segundos de antes y de después no pueden ser negativos, ni los dos cero")

if EN_COLAB and any(str(r).startswith("/content/drive") for r in (VIDEO, MODELO_ONNX, SALIDA)):
    from google.colab import drive

    drive.mount("/content/drive")


def pide_fichero(ruta, que):
    # El fichero de la ruta; si no está y estamos en Colab, se pide que se suba.
    ruta = Path(ruta)
    if ruta.is_file():
        return ruta
    if not EN_COLAB:
        raise SystemExit(f"no está {que}: {ruta}")
    from google.colab import files

    print(f"No encuentro {que} en {ruta}. Súbelo (para un partido entero, mejor desde Drive):")
    subidos = files.upload()
    if not subidos:
        raise SystemExit(f"sin {que} no se puede seguir")
    return Path(next(iter(subidos))).resolve()


video_path = pide_fichero(VIDEO, "el vídeo")
modelo_path = pide_fichero(MODELO_ONNX, "el modelo")
carpeta = Path(SALIDA) / video_path.stem
carpeta.mkdir(parents=True, exist_ok=True)
print("vídeo  :", video_path)
print("modelo :", modelo_path)
print("salida :", carpeta)

## 3 · El modelo

La ficha de `models/registry.yaml` dice cómo se ejecuta el modelo: el tamaño de la
ventana, el ritmo, el color, la normalización y el orden de las clases. Nada de eso está
dentro del `.onnx`, y equivocarse no da error: da basura. Por eso se comprueban el sha256
del fichero y la entrada y las salidas de la sesión.

In [ ]:
from fanalysis.ficha import load_ficha
from fanalysis.runner import OnnxClipRunner

ficha = load_ficha(REPO / "models" / "registry.yaml", "tdeed-snb")
corredor = OnnxClipRunner(ficha, modelo_path)  # comprueba el sha256 y la sesión

print(f"{ficha.name} {ficha.version}: ventanas de {ficha.clip_frames} frames "
      f"({ficha.window_s:g} s) a {ficha.sample_fps:g} fps, {ficha.width}×{ficha.height}")
print("clases    :", ", ".join(k for k in ficha.kinds if k != "normal_play"))
print("sha256    :", ficha.sha256[:16], "✓")
print("proveedor :", corredor.provider)
print("licencia  :", ficha.license)
if TIENE_GPU and corredor.provider != "CUDAExecutionProvider":
    # Seguir costaría horas: en CPU cada ventana tarda unos 25 s.
    raise SystemExit(
        "Hay GPU, pero onnxruntime abrió la sesión en la CPU. Mira arriba el error de "
        "onnxruntime (suele ser una librería de CUDA o cuDNN que no carga). Prueba con "
        "Entorno de ejecución → Reiniciar sesión y ejecuta otra vez desde la celda 1. "
        "Si quieres seguir en CPU de todas formas, cambia el entorno a CPU.")

## 4 · El vídeo

Antes de gastar GPU se comprueba que el vídeo sirve.

**Qué metraje entiende T-DEED.** Se entrenó con retransmisiones de televisión. Medido el
2026-10-19:
- con la cámara de estadio de un feed de scouting ve pases y conducciones a 0,8-0,9;
- con una panorámica amateur desde la grada, en la que los jugadores miden unos pocos
  píxeles, **no ve nada**, como si le dieran frames negros.

Con las grabaciones del soporte, usa la del **programa** (la cámara virtual que sale a
YouTube), no la panorámica en bruto.

- **Es un error:** que ffprobe no sepa leerlo, que sea vertical o que el tramo dure menos
  que una ventana. Con eso el análisis no vale nada.
- **Es un aviso:** otra proporción, menos fps que el modelo, poca resolución o falta de
  audio. Con eso el análisis vale, pero sale peor.

In [ ]:
from fanalysis.report import format_instant
from fanalysis.spotting import window_starts
from fanalysis.video import expected_frames, probe, validate

info = probe(video_path)
tramo = validate(info, ficha, start_s=INICIO_S, end_s=FIN_S)
ventanas = len(window_starts(expected_frames(tramo, ficha.sample_fps), ficha.clip_frames, SALTO_FRAMES))

print(f"{info.width}×{info.height} · {info.fps:.2f} fps · {info.codec} · "
      f"{'con' if info.has_audio else 'sin'} audio · {format_instant(info.duration_s)}")
print(f"tramo     : {format_instant(tramo.start_s)} → {format_instant(tramo.end_s)}")
print(f"ventanas  : {ventanas} de {ficha.window_s:g} s, una cada {SALTO_FRAMES / ficha.sample_fps:g} s")
for aviso in tramo.warnings:
    print("⚠️", aviso)

### ¿El modelo ve este vídeo?

Antes del análisis largo se pasan tres ventanas de prueba, repartidas por el tramo. Si en
ellas el modelo cree que «no pasa nada» casi todo el rato, como con una imagen vacía, el
análisis entero saldría vacío, y mejor saberlo ahora. Se usa la medida de arriba: con la
cámara de estadio, el fondo se queda en 0,44-0,46; con la panorámica amateur, en 0,76.

In [ ]:
from fanalysis.constants import BLIND_BACKGROUND_SHARE, BLIND_PROBE_WINDOWS
from fanalysis.spotting import activate, background_share, looks_blind, probe_starts
from fanalysis.video import read_window

IGNORAR_PRUEBA = False  #@param {type:"boolean"}

fondos = []
for inicio in probe_starts(tramo.start_s, tramo.end_s, ficha.window_s, BLIND_PROBE_WINDOWS):
    salidas = corredor.infer(read_window(info, ficha, inicio))
    fondos.append(background_share(activate(salidas["logits"], ficha.activation), ficha.kinds))
    print(f"ventana en {format_instant(inicio)}: fondo medio {fondos[-1]:.2f}")

if looks_blind(fondos):
    texto = (f"El modelo no está viendo este vídeo: el fondo medio queda por encima de "
             f"{BLIND_BACKGROUND_SHARE:g}. Seguramente es una panorámica o un plano muy "
             "abierto, y el análisis saldría vacío. Usa un plano más cerrado o la grabación "
             "del programa.")
    if not IGNORAR_PRUEBA:
        raise SystemExit(texto + " (Para seguir igualmente, marca IGNORAR_PRUEBA.)")
    print("⚠️", texto)
else:
    print("✓ el modelo ve el partido: se puede analizar")

## 5 · Detectar las acciones

ffmpeg saca los frames a 12,5 fps y a 796×448, y T-DEED los mira en ventanas de 8 s. Cada
frame se queda con la media de las ventanas que lo vieron.

Las puntuaciones se guardan en `puntuaciones.npz`, en la carpeta de salida. Si ya existen
para el mismo vídeo, el mismo tramo, el mismo modelo y el mismo salto, se reutilizan y
esta celda tarda un segundo. `RECALCULAR` obliga a pasar el modelo otra vez.

In [ ]:
import time

from fanalysis.spotting import Timeline, detect
from fanalysis.video import FfmpegFrameReader

npz = carpeta / "puntuaciones.npz"
linea = None
if npz.is_file() and not RECALCULAR:
    previa = Timeline.load(npz)
    misma = (
        previa.sha256 == ficha.sha256
        and previa.hop == SALTO_FRAMES
        and previa.source == video_path.name
        and previa.offset_s == tramo.start_s
        # el filtro `fps` de ffmpeg puede dar un frame de más o de menos en el borde
        and abs(previa.n_frames - expected_frames(tramo, ficha.sample_fps)) <= 1
    )
    if misma:
        linea = previa
        print("reutilizo las puntuaciones de", npz)
    else:
        print("hay puntuaciones guardadas, pero de otro vídeo, tramo, modelo o salto: se recalculan")

if linea is None:
    try:
        from tqdm.auto import tqdm

        barra = tqdm(total=ventanas, unit="ventana")
    except ImportError:
        barra = None
    inicio = time.perf_counter()

    def avance(hechas, _frames):
        if barra is not None:
            barra.update(hechas - barra.n)
        elif hechas % 20 == 0:
            print(f"{hechas}/{ventanas} ventanas")

    with FfmpegFrameReader(info, ficha, tramo) as lector:
        linea = detect(lector, corredor, ficha, hop=SALTO_FRAMES, offset_s=tramo.start_s,
                       name=video_path.name, on_window=avance)
        lector.check_complete()
    if barra is not None:
        barra.close()
    linea.save(npz)
    segundos = time.perf_counter() - inicio
    print(f"{linea.windows} ventanas en {segundos / 60:.1f} min "
          f"({segundos / linea.windows:.2f} s por ventana, {corredor.provider})")

## 6 · Las acciones

Una acción es un pico de su clase por encima del umbral, a 2 s o más de otro de su misma
clase. Todas van a `acciones.csv`.

In [ ]:
from collections import Counter

from fanalysis.report import action_rows, write_csv
from fanalysis.spotting import find_actions


def muestra(filas):
    # Una tabla: con pandas si está (en Colab, sí), y si no, en texto.
    if not filas:
        print("(nada)")
        return
    try:
        import IPython.display
        import pandas as pd

        IPython.display.display(pd.DataFrame(filas))
    except ImportError:
        for fila in filas:
            print(fila)


acciones = find_actions(linea, threshold=UMBRAL_ACCIONES)
write_csv(action_rows(acciones), carpeta / "acciones.csv")
recuento = Counter(a.kind for a in acciones)
print(f"{len(acciones)} acciones por encima de {UMBRAL_ACCIONES:g} en "
      f"{format_instant(linea.end_s - linea.offset_s)} de vídeo:")
muestra([{"clase": clase, "veces": veces} for clase, veces in recuento.most_common()])

## 7 · Los goles

Cada pico de **GOAL** es un candidato, y pasa dos comprobaciones:

1. **Duplicados.** Dos picos a menos de 20 s son el mismo gol, porque entre dos goles hay
   celebración y saque de centro. Se queda el más seguro, y el otro va en `fundidos`.
2. **Remate.** Se busca un tiro o un cabezazo en los 2,5 s anteriores. Con remate, el gol
   queda `confirmado`; sin él, queda `sin_remate`.

En la gráfica, los goles confirmados van rellenos y los que no tienen remate, huecos.

In [ ]:
import matplotlib.pyplot as plt

from fanalysis.goals import find_goals
from fanalysis.plots import plot_timeline
from fanalysis.report import goal_row

goles = find_goals(linea, threshold=UMBRAL_GOL, finish_threshold=UMBRAL_ACCIONES)
print(f"{len(goles)} goles candidatos:")
muestra([goal_row(g) for g in goles])
plot_timeline(linea, goles, threshold=UMBRAL_GOL)
plt.show()

## 8 · ¿Están todos? (opcional)

Si sabes cuándo hubo gol, apúntalo aquí y el cuaderno calcula la **precisión** y el
**recall** con ±2 s de tolerancia. Es la misma vara con la que se miden los demás eventos
del proyecto.

Los instantes van en tiempo del vídeo, separados por comas o uno por línea: `264`,
`04:24` o `1:04:24`. Lo que haya fuera del tramo analizado no cuenta.

El recall de T-DEED sobre nuestras cámaras **no se ha medido nunca**, así que esta celda
es la que lo mide.

In [ ]:
from fanalysis.goals import STATUS_NO_FINISH, match_goals, parse_marks
from fanalysis.report import match_row

GOLES_REALES = ""  #@param {type:"string"}

medidas = {}
reales = [t for t in parse_marks(GOLES_REALES) if tramo.start_s <= t <= tramo.end_s]
if reales:
    confirmados = [g.t_s for g in goles if g.status != STATUS_NO_FINISH]
    for nombre, detectados in (("todos los candidatos", [g.t_s for g in goles]),
                               ("solo con remate", confirmados)):
        medidas[nombre] = match_row(match_goals(detectados, reales))
    muestra([{"goles": nombre, **fila} for nombre, fila in medidas.items()])
else:
    print("Sin goles anotados: no se mide. Para medir, escribe sus instantes en GOLES_REALES.")

## 9 · Revisión

El sistema sugiere y una persona decide (ADR 0013 de football-ai). Aquí tienes el
fotograma de cada gol, en su instante y un segundo después. Para quitar o poner goles,
escribe sus números en `QUITAR` y `PONER`, separados por comas. Con `PONER` entra también
un gol sin remate.

In [ ]:
from fanalysis.clips import run_or_fail, snapshot_command

fotos = carpeta / "fotogramas"
fotos.mkdir(exist_ok=True)
if goles:
    figura, ejes = plt.subplots(len(goles), 2, figsize=(10, 2.9 * len(goles)), squeeze=False)
    for fila, gol in zip(ejes, goles, strict=True):
        for eje, desfase in zip(fila, (0.0, 1.0), strict=True):
            # Hasta un segundo antes del final: buscar en el último frame a veces no saca nada.
            instante = min(gol.t_s + desfase, info.duration_s - 1.0)
            foto = fotos / f"gol_{gol.number:02d}_{desfase:+.0f}s.png"
            foto.unlink(missing_ok=True)
            run_or_fail(snapshot_command(video_path, foto, t_s=instante), "sacar un fotograma")
            if foto.is_file():
                eje.imshow(plt.imread(foto))
            eje.set_axis_off()
            eje.set_title(f"gol {gol.number} · {format_instant(instante)} · {gol.status}",
                          fontsize=9, loc="left")
    figura.tight_layout()
    plt.show()
else:
    print("No hay goles que revisar.")

In [ ]:
from fanalysis.goals import parse_numbers, select_for_highlights

QUITAR = ""  #@param {type:"string"}
PONER = ""  #@param {type:"string"}

elegidos = select_for_highlights(goles, require_finish=EXIGIR_REMATE,
                                 exclude=parse_numbers(QUITAR), include=parse_numbers(PONER))
fuera = [g.number for g in goles if g not in elegidos]
print(f"Van a los highlights {len(elegidos)} de {len(goles)} goles: {[g.number for g in elegidos]}")
if fuera:
    print(f"Se quedan fuera: {fuera}. Para meter alguno, pon su número en PONER.")

## 10 · Clips y highlights

Cada gol se corta con **20 s antes y 10 s después**, o lo que hayas puesto arriba.

- Si un gol está pegado al principio o al final del vídeo, el clip se queda con lo que
  hay, y la tabla lo marca en `antes_recortado` o `despues_recortado`.
- Dos goles cuyos clips se pisan salen en un único clip.

El corte se recodifica para que caiga justo en el frame, y cada clip se comprueba: tiene
que durar lo que se pidió.

In [ ]:
from fanalysis.clips import build_highlights, cut_clips, plan_clips
from fanalysis.report import clip_row

carpeta_clips = carpeta / "clips"
# Los clips de una pasada anterior se borran: si no, se mezclarían con los de esta.
for viejo in [*carpeta_clips.glob("gol_*.mp4"), *carpeta_clips.glob("goles_*.mp4")]:
    viejo.unlink()

clips, highlights = [], None
if elegidos:
    planes = plan_clips(elegidos, duration_s=info.duration_s,
                        pre_s=SEGUNDOS_ANTES, post_s=SEGUNDOS_DESPUES)
    clips = cut_clips(video_path, planes, carpeta_clips, has_audio=info.has_audio)
    highlights = build_highlights(clips, carpeta / "highlights.mp4")
    muestra([clip_row(c) for c in clips])
    print(f"highlights: {highlights} ({format_instant(sum(c.duration_s for c in clips))})")
else:
    print("Ningún gol pasó la revisión: no hay clips ni highlights.")

## 11 · Ver un clip (opcional)

Pon el número de clip, empezando en 1. Se hace una copia ligera de 640 px para verla
aquí sin cargar el original.

In [ ]:
from fanalysis.clips import preview_command

VER_CLIP = 0  #@param {type:"integer"}

if 1 <= VER_CLIP <= len(clips):
    vista = carpeta / "vista_previa.mp4"
    run_or_fail(preview_command(clips[VER_CLIP - 1].path, vista), "hacer la vista previa")
    try:
        import IPython.display
    except ImportError:  # fuera de un cuaderno no hay dónde enseñarla
        print("vista previa en", vista)
    else:
        datos = base64.b64encode(vista.read_bytes()).decode()
        IPython.display.display(IPython.display.HTML(
            f'<video controls width="640" src="data:video/mp4;base64,{datos}"></video>'))
elif VER_CLIP:
    print(f"Hay {len(clips)} clips: elige uno entre 1 y {len(clips)}.")

## 12 · El informe

`informe.json` lleva el vídeo, el modelo, los parámetros, cada gol con lo que se
comprobó, los clips, la medida si la hubo y los avisos. Con eso, un resultado se puede
explicar dentro de un mes.

In [ ]:
from fanalysis.report import write_json

informe = {
    "video": {"fichero": video_path.name, "duracion_s": round(info.duration_s, 2),
              "ancho": info.width, "alto": info.height, "fps": round(info.fps, 3),
              "audio": info.has_audio, "rotacion": info.rotation},
    "tramo": {"desde": format_instant(tramo.start_s), "hasta": format_instant(tramo.end_s)},
    "avisos": list(tramo.warnings),
    "modelo": {"nombre": ficha.name, "version": ficha.version, "sha256": ficha.sha256,
               "proveedor": corredor.provider, "licencia": ficha.license},
    "parametros": {"umbral_gol": UMBRAL_GOL, "umbral_acciones": UMBRAL_ACCIONES,
                   "exigir_remate": EXIGIR_REMATE, "segundos_antes": SEGUNDOS_ANTES,
                   "segundos_despues": SEGUNDOS_DESPUES, "salto_frames": SALTO_FRAMES,
                   "ventanas": linea.windows},
    "acciones": dict(recuento.most_common()),
    "goles": [goal_row(g) for g in goles],
    "en_highlights": [g.number for g in elegidos],
    "clips": [clip_row(c) for c in clips],
    "highlights": highlights.name if highlights else None,
    "medida": medidas or None,
}
write_json(informe, carpeta / "informe.json")
print("Todo en", carpeta)
for fichero in sorted(carpeta.rglob("*")):
    if fichero.is_file():
        print(f"  {fichero.relative_to(carpeta)}  ({fichero.stat().st_size / 1e6:.1f} MB)")

if EN_COLAB and not str(carpeta).startswith("/content/drive") and highlights:
    from google.colab import files

    files.download(str(highlights))
    files.download(str(carpeta / "informe.json"))